# 05 Statistical Tests

Google Colab notebook version.

In [ ]:

# ============================================================
# STATISTICAL TESTS
# ============================================================

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import mean_squared_error

def rmse(y, yhat):
    return float(np.sqrt(mean_squared_error(y, yhat)))

def dm_test_one_sided(d, L=24):
    d = np.asarray(d).astype(float)
    n = len(d)
    dbar = d.mean()
    d0 = d - dbar

    gamma0 = np.mean(d0 * d0)
    lrv = gamma0

    for k in range(1, L + 1):
        cov = np.mean(d0[k:] * d0[:-k])
        w = 1.0 - k / (L + 1.0)
        lrv += 2.0 * w * cov

    if lrv <= 1e-12:
        return np.nan, np.nan

    dm = dbar / np.sqrt(lrv / n)
    p = 1.0 - stats.norm.cdf(dm)
    return float(dm), float(p)

def block_bootstrap_ci_delta_rmse(y, y_stam, y_base, block=24, B=3000, seed=0):
    rng = np.random.default_rng(seed)
    y = np.asarray(y)
    stam = np.asarray(y_stam)
    base = np.asarray(y_base)

    n = len(y)
    idx = np.arange(n)
    deltas = np.empty(B, dtype=float)

    for b in range(B):
        sampled = []
        while len(sampled) < n:
            start = rng.integers(0, n - block + 1)
            sampled.extend(idx[start:start + block])
        sampled = np.array(sampled[:n])
        deltas[b] = rmse(y[sampled], base[sampled]) - rmse(y[sampled], stam[sampled])

    return np.quantile(deltas, [0.025, 0.975])

def holm_adjust_pvalues(pvals):
    p = pd.Series(pvals).astype(float)
    m = len(p)

    order = np.argsort(p.values)
    ranked = p.values[order]

    adj = np.empty(m, dtype=float)

    for i in range(m):
        adj[i] = (m - i) * ranked[i]

    for i in range(1, m):
        adj[i] = max(adj[i], adj[i - 1])

    adj = np.minimum(adj, 1.0)

    out = pd.Series(index=p.index, dtype=float)
    out.iloc[order] = adj
    return out

def extract_actual_pred(df):
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]

    if {"Actual", "Prediction"}.issubset(df.columns):
        return df["Actual"].to_numpy(dtype=float), df["Prediction"].to_numpy(dtype=float)

    if {"Actual_GSR", "Predicted_GSR"}.issubset(df.columns):
        return df["Actual_GSR"].to_numpy(dtype=float), df["Predicted_GSR"].to_numpy(dtype=float)

    raise ValueError(
        "CSV must contain either columns (Actual, Prediction) or "
        "(Actual_GSR, Predicted_GSR). "
        f"Found columns: {list(df.columns)}"
    )

def run_stam_vs_baselines_significance(
    model_files,
    stam_name="CNN-BiLSTM-STAM",
    include_night=True,
    dm_lag=24,
    boot_block=24,
    boot_B=3000,
    two_sided=True,
):
    data = {}

    for model_name, path in model_files.items():
        df = pd.read_csv(path)
        y_true, y_pred = extract_actual_pred(df)
        data[model_name] = (y_true.reshape(-1), y_pred.reshape(-1))

    y_true_ref, y_stam = data[stam_name]
    n_ref = len(y_true_ref)

    mismatched = []
    for model_name, (y_true, _) in data.items():
        if len(y_true) != n_ref:
            mismatched.append((model_name, f"length mismatch: {len(y_true)} vs {n_ref}"))
            continue

        max_abs_diff = float(np.max(np.abs(y_true - y_true_ref)))
        if max_abs_diff > 1e-6:
            mismatched.append((model_name, f"actual mismatch: max diff {max_abs_diff}"))




    if include_night:
        mask = np.ones_like(y_true_ref, dtype=bool)
    else:
        mask = np.abs(y_true_ref) > 1e-2

    y_true_eval = y_true_ref[mask]
    y_stam_eval = y_stam[mask]

    rmse_stam = rmse(y_true_eval, y_stam_eval)
    results = []

    alternative = "two-sided" if two_sided else "greater"

    for model_name, (y_true, y_pred) in data.items():
        if model_name == stam_name:
            continue

        y_base_eval = y_pred[mask]
        rmse_base = rmse(y_true_eval, y_base_eval)
        delta_rmse = rmse_base - rmse_stam

        e_stam = y_true_eval - y_stam_eval
        e_base = y_true_eval - y_base_eval
        d_sq = (e_base ** 2) - (e_stam ** 2)

        t_res = stats.ttest_1samp(d_sq, 0.0, alternative=alternative)
        w_res = stats.wilcoxon(d_sq, zero_method="wilcox", alternative=alternative)

        dm_stat, dm_p = dm_test_one_sided(d_sq, L=dm_lag)

        ci_low, ci_high = block_bootstrap_ci_delta_rmse(
            y_true_eval,
            y_stam_eval,
            y_base_eval,
            block=boot_block,
            B=boot_B,
            seed=0,
        )

        results.append({
            "Baseline_model": model_name,
            "RMSE_STAM": rmse_stam,
            "RMSE_baseline": rmse_base,
            "Delta_RMSE_baseline_minus_STAM": delta_rmse,
            "t_test_p": float(t_res.pvalue),
            "wilcoxon_p": float(w_res.pvalue),
            f"DM_p_L{dm_lag}": float(dm_p),
            f"Bootstrap95CI_DeltaRMSE_block{boot_block}": f"[{ci_low:.4f}, {ci_high:.4f}]",
            "Mean_d_sq": float(np.mean(d_sq)),
        })

    res_df = pd.DataFrame(results).sort_values(
        "Delta_RMSE_baseline_minus_STAM",
        ascending=False,
    )

    res_df["wilcoxon_p_holm"] = holm_adjust_pvalues(
        res_df.set_index("Baseline_model")["wilcoxon_p"]
    ).values

    dm_col = f"DM_p_L{dm_lag}"
    res_df["DM_p_holm"] = holm_adjust_pvalues(
        res_df.set_index("Baseline_model")[dm_col]
    ).values

    res_df.to_csv(output_csv, index=False)

    print("\n=== Significance: STAM vs all baselines ===")
    print(res_df.to_string(index=False))

    return res_df


In [ ]:
# Example use: upload all baseline prediction CSV files first
MODEL_FILES = {
    # "CNN-BiLSTM-BAM": "/content/CNN-BiLSTM-BAM.csv",
    # "CNN-BiLSTM-CAM": "/content/CNN-BiLSTM-CAM.csv",
    # "CNN-BiLSTM-LAM": "/content/CNN-BiLSTM-LAM.csv",
    # "CNN-BiLSTM-MHAM": "/content/CNN-BiLSTM-MHAM.csv",
    # "CNN-BiLSTM-TAM": "/content/CNN-BiLSTM-TAM.csv",
    # "CNN-BiLSTM-STAM": "/content/GSR_actual_vs_predicted.csv",
}

# stats_df = run_stam_vs_baselines_significance(
#     model_files=MODEL_FILES,
#     stam_name="CNN-BiLSTM-STAM",
#     include_night=True,
#     dm_lag=24,
#     boot_block=24,
#     boot_B=3000,
#     output_csv="/content/STAM_vs_all_models_significance.csv",
#     two_sided=True,
# )